In [1]:
pip install pytorch-ssim


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for pytorch-ssim: filename=pytorch_ssim-0.1-py3-none-any.whl size=2050 sha256=ceefa804e54624e17a6114ef7d483dfc15ae14d3584d76fab313329387c112d5
  Stored in directory: c:\users\ashut\appdata\local\pip\cache\wheels\54\a0\11\99f86224e71729ed9ef0c4ffe1b795807ad5f44bde19bc66f9
Successfully built pytorch-ssim
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install torch torchvision torchaudio

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   -------------------- ------------------- 0.8/1.6 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 13.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 15.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
##### testing cuda 

import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")


CUDA Available: False
GPU Device Name: No GPU found


In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
pip uninstall torch torchvision torchaudio -y


Found existing installation: torch 2.6.0
Uninstalling torch-2.6.0:
  Successfully uninstalled torch-2.6.0
Found existing installation: torchvision 0.21.0
Uninstalling torchvision-0.21.0:
  Successfully uninstalled torchvision-0.21.0
Found existing installation: torchaudio 2.6.0
Uninstalling torchaudio-2.6.0:
  Successfully uninstalled torchaudio-2.6.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [9]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 5.6 MB/s eta 0:07:20
     ---------------------------------------- 0.0/2.4 GB 11.2 MB/s eta 0:03:39
     ---------------------------------------- 0.0/2.4 GB 9.6 MB/s eta 0:04:16
     ---------------------------------------- 0.0/2.4 GB 10.8 MB/s eta 0:03:47
     ---------------------------------------- 0.0/2.4 GB 9.1 MB/s eta 0:04:28
     ---------------------------------------- 0.0/2.4 GB 9.7 MB/s eta 0:04:11
     ---------------------------------------- 0.0/2.4 GB 9.5 MB/s eta 0:04:16
     ---------------------------------------- 0.0/2.4 GB 8.2 MB/s eta 0:04:57
     ---------------------------------------- 0.0/2.4 GB 7.5 MB/s eta 0:05:27
     ---------------------------------------- 0.0/2.4 GB 7.9 MB/s eta 0:05:08
     ---------------------------------------- 0.0/2.4 GB 8.8 MB/s eta 0:04:36
     -----


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


CUDA Available: True
Number of GPUs: 1
GPU Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
'''
ERGSAN MODEL

'''



import os
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import pytorch_ssim



In [3]:
HQ_FOLDER = r"C:\Users\ashut\Desktop\ElementaryCQT"  # Change this
LQ_FOLDER = r"C:\Users\ashut\Desktop\ElementaryCQT_LOWRES"   # Change this

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (4160368914.py, line 1)

In [14]:
import os
import glob
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ---- STEP 1: CUSTOM DATASET ----
class CustomDataset(Dataset):
    def __init__(self, hqFolder, lqFolder, imgSize=(256, 256)):  # Fixed image size
        # Recursively find all image files
        self.hqImages = sorted(glob.glob(os.path.join(hqFolder, "**", "*.png"), recursive=True) +
                               glob.glob(os.path.join(hqFolder, "**", "*.jpg"), recursive=True))

        self.lqImages = sorted(glob.glob(os.path.join(lqFolder, "**", "*.png"), recursive=True) +
                               glob.glob(os.path.join(lqFolder, "**", "*.jpg"), recursive=True))

        assert len(self.hqImages) == len(self.lqImages), "Mismatch between HQ and LQ image counts"

        # Transform: Resize all images to the same dimensions and convert to tensors
        self.transform = transforms.Compose([
            transforms.Resize(imgSize),  # Resize to fixed size
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.hqImages)

    def __getitem__(self, index):
        hqPath = self.hqImages[index]
        lqPath = self.lqImages[index]

        hqImg = Image.open(hqPath).convert('RGB')
        lqImg = Image.open(lqPath).convert('RGB')

        hqImg = self.transform(hqImg)
        lqImg = self.transform(lqImg)

        return lqImg, hqImg

# Create DataLoader
trainDataset = CustomDataset(HQ_FOLDER, LQ_FOLDER, imgSize=(256, 256))  # Set desired image size
trainLoader = DataLoader(trainDataset, batch_size=16, shuffle=True)

# Test DataLoader
for lrImg, hrImg in trainLoader:
    print("Batch Loaded:", lrImg.shape, hrImg.shape)  # Expect (Batch, 3, 256, 256)
    break


Batch Loaded: torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])


In [15]:
# ---- STEP 2: DEFINE Real-ESRGAN MODEL ----
class RRDB(nn.Module):
    def __init__(self, inChannels):
        super(RRDB, self).__init__()
        self.conv1 = nn.Conv2d(inChannels, inChannels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(inChannels, inChannels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(inChannels, inChannels, kernel_size=3, padding=1)
        self.activation = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        residual = x
        x = self.activation(self.conv1(x))
        x = self.activation(self.conv2(x))
        x = self.conv3(x)
        return x + residual

class RealESRGAN(nn.Module):
    def __init__(self, numRRDB=10, upscaleFactor=4):
        super(RealESRGAN, self).__init__()
        self.inputConv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.rrdbBlocks = nn.Sequential(*[RRDB(32) for _ in range(numRRDB)])
        self.outputConv = nn.Conv2d(32, 3 * (upscaleFactor ** 2), kernel_size=3, padding=1)
        self.pixelShuffle = nn.PixelShuffle(upscaleFactor)

    def forward(self, x):
        x = self.inputConv(x)
        x = self.rrdbBlocks(x)
        x = self.outputConv(x)
        x = self.pixelShuffle(x)
        return x

# Initialize model
model = RealESRGAN().cuda()

In [16]:
# ---- STEP 3: DEFINE LOSS & OPTIMIZER ----
class CharbonnierLoss(nn.Module):
    def __init__(self, epsilon=1e-6):
        super(CharbonnierLoss, self).__init__()
        self.epsilon = epsilon

    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.epsilon))

ssimLoss = pytorch_ssim.SSIM()
charbLoss = CharbonnierLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)


In [17]:
# ---- STEP 4: TRAINING LOOP ----
numEpochs = 20  # Adjust as needed
for epoch in range(numEpochs):
    epochLoss = 0
    for lrImgs, hrImgs in tqdm(trainLoader):
        lrImgs, hrImgs = lrImgs.cuda(), hrImgs.cuda()

        optimizer.zero_grad()
        preds = model(lrImgs)

        loss1 = charbLoss(preds, hrImgs)
        loss2 = 1 - ssimLoss(preds, hrImgs)
        loss = loss1 + loss2  # Combined loss

        loss.backward()
        optimizer.step()

        epochLoss += loss.item()

    print(f"Epoch {epoch+1}/{numEpochs} - Loss: {epochLoss:.4f}")

# Save trained model
torch.save(model.state_dict(), "RealESRGAN.pth")
print("Model saved!")

  0%|          | 0/23750 [00:01<?, ?it/s]


RuntimeError: The size of tensor a (1024) must match the size of tensor b (256) at non-singleton dimension 3

In [2]:
pip install torchmetrics


   ---------------------------------------- 0.0/931.7 kB ? eta -:--:--
   --------------------------------------- 931.7/931.7 kB 10.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from PIL import Image
from torchmetrics.image import StructuralSimilarityIndexMeasure

# ---- STEP 1: CUSTOM DATASET ----
class CustomDataset(Dataset):
    def __init__(self, hqFolder, lqFolder, lqSize=(120, 120), hqSize=(200, 200)):  
        self.hqImages = sorted(glob.glob(os.path.join(hqFolder, "**", "*.png"), recursive=True) +
                               glob.glob(os.path.join(hqFolder, "**", "*.jpg"), recursive=True))
        self.lqImages = sorted(glob.glob(os.path.join(lqFolder, "**", "*.png"), recursive=True) +
                               glob.glob(os.path.join(lqFolder, "**", "*.jpg"), recursive=True))
        assert len(self.hqImages) == len(self.lqImages), "Mismatch between HQ and LQ image counts"

        self.hqTransform = transforms.Compose([
            transforms.Resize(hqSize),  
            transforms.ToTensor()
        ])
        self.lqTransform = transforms.Compose([
            transforms.Resize(lqSize),  
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.hqImages)

    def __getitem__(self, index):
        hqImg = Image.open(self.hqImages[index]).convert('RGB')
        lqImg = Image.open(self.lqImages[index]).convert('RGB')
        return self.lqTransform(lqImg), self.hqTransform(hqImg)

# Create DataLoader
HQ_FOLDER = r"C:\Users\ashut\Desktop\ElementaryCQT"
LQ_FOLDER = r"C:\Users\ashut\Desktop\ElementaryCQT_LOWRES"

trainDataset = CustomDataset(HQ_FOLDER, LQ_FOLDER)  
trainLoader = DataLoader(trainDataset, batch_size=16, shuffle=True)

# ---- STEP 2: DEFINE Real-ESRGAN MODEL ----
class RRDB(nn.Module):
    def __init__(self, inChannels):
        super(RRDB, self).__init__()
        self.conv1 = nn.Conv2d(inChannels, inChannels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(inChannels, inChannels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(inChannels, inChannels, kernel_size=3, padding=1)
        self.activation = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        residual = x
        x = self.activation(self.conv1(x))
        x = self.activation(self.conv2(x))
        x = self.conv3(x)
        return x + residual

class RealESRGAN(nn.Module):
    def __init__(self, numRRDB=10, upscaleFactor=200 // 120):  #upscalefactor here
        super(RealESRGAN, self).__init__()
        self.inputConv = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.rrdbBlocks = nn.Sequential(*[RRDB(32) for _ in range(numRRDB)])
        self.outputConv = nn.Conv2d(32, 3 * (upscaleFactor ** 2), kernel_size=3, padding=1)
        self.pixelShuffle = nn.PixelShuffle(upscaleFactor)

    def forward(self, x):
        x = self.inputConv(x)
        x = self.rrdbBlocks(x)
        x = self.outputConv(x)
        x = self.pixelShuffle(x)
        return x

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RealESRGAN().to(device)

# ---- STEP 3: DEFINE LOSS & OPTIMIZER ----
class CharbonnierLoss(nn.Module):
    def __init__(self, epsilon=1e-3):
        super(CharbonnierLoss, self).__init__()
        self.epsilon = epsilon

    def forward(self, preds, targets):
        return torch.mean(torch.sqrt((preds - targets) ** 2 + self.epsilon ** 2))

# Instantiate loss functions & optimizer
charbLoss = CharbonnierLoss()
ssimLoss = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ---- STEP 4: TRAINING LOOP ----
numEpochs = 20  
for epoch in range(numEpochs):
    model.train()
    epochLoss = 0

    for lrImgs, hrImgs in tqdm(trainLoader):
        lrImgs, hrImgs = lrImgs.to(device), hrImgs.to(device)

        optimizer.zero_grad()
        preds = model(lrImgs)

        # Ensure predictions match HR image size
        preds = F.interpolate(preds, size=(200, 200), mode='bilinear', align_corners=False)

        # Compute losses
        loss1 = charbLoss(preds, hrImgs)
        loss2 = 1 - ssimLoss(preds, hrImgs)
        loss = loss1 + loss2

        loss.backward()
        optimizer.step()

        epochLoss += loss.item()

    print(f"Epoch {epoch+1}/{numEpochs} - Loss: {epochLoss:.4f}")

# Save trained model
torch.save(model.state_dict(), "RealESRGAN.pth")
print("Model saved!")


c:\Users\ashut\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 23750/23750 [2:22:13<00:00,  2.78it/s]  


Epoch 1/20 - Loss: 524.3329


100%|██████████| 23750/23750 [1:50:10<00:00,  3.59it/s]


Epoch 2/20 - Loss: 415.8654


100%|██████████| 23750/23750 [1:46:49<00:00,  3.71it/s]


Epoch 3/20 - Loss: 414.9157


... (16 outputs left out)

100%|██████████| 23750/23750 [1:46:52<00:00,  3.70it/s]


Epoch 12/20 - Loss: 403.9637


  0%|          | 22/23750 [00:06<1:52:45,  3.51it/s]


KeyboardInterrupt: 

In [2]:
import os
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

def get_image_info_parallel(root_dir, output_csv, max_workers=16):
    """
    Efficiently traverses directories, reads image dimensions in parallel threads,
    prints progress, and saves info to CSV.
    """
    image_data = []
    lock = threading.Lock()  # To safely update shared variables from multiple threads
    processed_count = 0

    # Collect all image paths first
    image_paths = []
    for dirpath, _, filenames in os.walk(root_dir):
        for file in filenames:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
                image_paths.append((dirpath, file))

    total_images = len(image_paths)
    print(f"Found {total_images} images. Processing...")

    def process_image(dirpath_file):
        nonlocal processed_count
        dirpath, file = dirpath_file
        image_path = os.path.join(dirpath, file)
        try:
            with Image.open(image_path) as img:
                width, height = img.size
            relative_dir = os.path.relpath(dirpath, root_dir)

            # Update and print progress safely
            with lock:
                processed_count += 1
                print(f"[{processed_count}/{total_images}] Processed: {os.path.join(relative_dir, file)}")

            return [relative_dir, file, f"{width}x{height}"]
        except Exception as e:
            with lock:
                processed_count += 1
                print(f"[{processed_count}/{total_images}] Failed to process: {image_path}, error: {e}")
            return None

    # Multithreaded processing
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_image = {executor.submit(process_image, dp_file): dp_file for dp_file in image_paths}

        for future in as_completed(future_to_image):
            result = future.result()
            if result:
                image_data.append(result)

    # Convert and save CSV
    df = pd.DataFrame(image_data, columns=["Directory Name", "Image Name", "Image Dimensions"])
    df.to_csv(output_csv, index=False)
    print(f"\n✅ Image information saved to {output_csv}")

# === Run the script ===
dataset_directory = r"C:\Users\ashut\Documents\GitHub\2dshapedetection-major\ElementaryCQT"
output_file = r"C:\Users\ashut\Documents\GitHub\2dshapedetection-major\ElementaryCQT\image_info.csv"

get_image_info_parallel(dataset_directory, output_file, max_workers=16)  # Adjust workers if needed


Found 380000 images. Processing...
[1/380000] Processed: Circle\Circle_Chord\circ_chr_1005.png
[2/380000] Processed: Circle\Circle_Chord\circ_chr_101.png
[3/380000] Processed: Circle\Circle_Chord\circ_chr_1003.png
[4/380000] Processed: Circle\Circle_Chord\circ_chr_1007.png
[5/380000] Processed: Circle\Circle_Chord\circ_chr_1000.png
[6/380000] Processed: Circle\Circle_Chord\circ_chr_0.png
[7/380000] Processed: Circle\Circle_Chord\circ_chr_1004.png
[8/380000] Processed: Circle\Circle_Chord\circ_chr_1008.png
[9/380000] Processed: Circle\Circle_Chord\circ_chr_1006.png
[10/380000] Processed: Circle\Circle_Chord\circ_chr_1002.png
[11/380000] Processed: Circle\Circle_Chord\circ_chr_1.png
[12/380000] Processed: Circle\Circle_Chord\circ_chr_10.png
[13/380000] Processed: Circle\Circle_Chord\circ_chr_1016.png
[14/380000] Processed: Circle\Circle_Chord\circ_chr_1009.png
[15/380000] Processed: Circle\Circle_Chord\circ_chr_1011.png
[16/380000] Processed: Circle\Circle_Chord\circ_chr_1010.png
[17/380

In [ ]:
import os
import cv2
import pandas as pd

def get_image_info(root_dir, output_excel):
    """
    Traverses through all subdirectories in root_dir, records image details (directory, name, dimensions),
    and exports the data into an Excel file.
    """
    image_data = []  # List to store image details
    
    for dirpath, _, filenames in os.walk(root_dir):
        for file in filenames:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
                image_path = os.path.join(dirpath, file)

                # Read the image to get dimensions
                image = cv2.imread(image_path)
                if image is None:
                    print(f"Skipping unreadable image: {image_path}")
                    continue
                
                height, width, _ = image.shape
                
                # Extract folder structure relative to root_dir
                relative_dir = os.path.relpath(dirpath, root_dir)

                # Store data (Directory Name, Image Name, Dimensions)
                image_data.append([relative_dir, os.path.basename(file), f"{width}x{height}"])

    # Convert to Pandas DataFrame
    df = pd.DataFrame(image_data, columns=["Directory Name", "Image Name", "Image Dimensions"])

    # Save to Excel
    df.to_excel(output_excel, index=False)
    print(f"Image information saved to {output_excel}")

# Set your dataset directory and output file
dataset_directory = "/Users/ashutosh/ProjectMajor/BasicGeometry_200x200_HIGHRES"
output_file = "/Users/ashutosh/ProjectMajor/image_info.xlsx"

# Run the script
get_image_info(dataset_directory, output_file)
